# Posterior ratio test for recurrent convergence

In preliminary analyses, we have observed an inflation of infinite sites model (ISM) violations (i.e., somatic variants that evolve more than once within the population of hematopoietic progenitors). While the stochastic character mapping (SCM) model is permissive of such complex histories, violations of infinite sites are expected to be distributed (*somewhat*) randomly across the tree. Note that an exception to this expectation are lineages with elevated evolutionary rates which would subseqently be inflated for mutations with complex histories. 

## 1. Background

![SCM violations](figs/recurrent_violations.jpg)

In the figure above, we see three clades (color-coded) with two tips that display excess sharing of convergent mutations. The top panel is the maximum likelihood phylogenetic tree inferred using all somatic single nucleotide variants (scale bar is measured in substitutions per site). The bottom reflects principle component analysis performed on an $N \times K$ matrix, where $N$ are clones and $K$ is a boolean reflecting presence/absence of a somatic single nucleotide variant that has beem mapped to > one branch. Note that in this example, variants are assigned to any branch where $\mathrm{Pr(GT_{descendent}) \ge 0.95}$, $\mathrm{Pr(GT_{ancestor}) \ge 0.95}$, and $\mathrm{GT_{descendent} \ne \mathrm{GT_{ancestor}}}$.

A few patterns emerge when looking at pairs with repeated ISM-violation sharing. *First*, node supports for their alliance is sufficiently strong--supporting their phylogenetic alliance. *Second*, the edge length of the last common ancestor's lineage is often short. Given the continuous-time Markov model (approximated using a Poisson process) at the core of SCM, this short edge length depletes the "temporal" window in which a mutation can evolve. While this is not displayed on the figure, the *third* observation is that the SCM-approximated posterior probability of the derived genotype is often $0.95 > \mathrm{Pr} ≥ 0.50$. Thus, quantitative support for a single origin of the derived genotype state is present, though below the threshold for confident assignment.

## 2. Approach

Given a vector of genotype states $S = \{A, B, C\}$, the objective is to compare evidence for two competing hypotheses:
- $H_0$: Somatic single nucleotide variant $S_j$ was present in the last common ancestor to *Clone 1* and *Clone 2* and was inherited by both descendent lineages.
- $H_1$: Somatic single nucleotide variant $S_i$ was present in the last common ancestor, and $S_j$ evolved independently in *Clone 1* and *Clone 2*.

To quantitatively compare these hypotheses, we will quantify the posterior probability ratio $Pr(H_0 | \vec{\theta}) / Pr(H_1 | \vec{\theta})$, where $\vec{\theta}$ is a placeholder for the following parameters:
- $\tau$: Maximum-likelihood phylogenetic tree with informative branch lengths $v$
- $\boldsymbol{Q}$: Substitution matrix jointly fit to $\tau$
- $S$: All possible genotype states given alleles observed at the locus

Along any arbitrary branch $v$, the probability of a genotype state transition between states $S_i$ and $S_j$ is given by $p_{ij} = e^{\boldsymbol{Q}v}$. From SCM, we approximate the posterior probability of the last common ancestor possessing the derived genotype state $S_j$ and can use this an informed prior, 

$$
\mathrm{Pr}(\vec{\theta}) = \mathrm{Pr_{SCM}}(GT_{LCA} = S_j).
$$ 

Thus, we can compute the posterior odds of $H_1 / H_0$ as the following beginning with Bayes theorem:

$$
\begin{align*}
\frac{\mathrm{Pr}(H_1|\vec{\theta})}
     {\mathrm{Pr}(H_0|\vec{\theta})}
        &= \frac{\mathrm{Pr}(\vec{\theta}|H_1)\times\mathrm{Pr_{SCM}}(GT_{LCA} \ne S_j)}
                {\mathrm{Pr}(\vec{\theta}|H_0)\times\mathrm{Pr_{SCM}}(GT_{LCA} = S_j)} \\
        &= \frac{p_{ij}\times\mathrm{Pr_{SCM}}(GT_{LCA} \ne S_j)}
                {p_{ij}\times\mathrm{Pr_{SCM}}(GT_{LCA} = S_j)}
\end{align*}
$$

Given that there are multiple genotype state transition paths to yeild state $S_j$ (particularly when $S_j$ is not well-resolved), we can sum across each genotype state transition path and take the product across independently evolving lineages. This most comprehensively approximates the likelihood of the data under each hypothesis (*equation 1*):

$$
\begin{align}
\frac{\mathrm{Pr}(H_1|\vec{\theta})}
     {\mathrm{Pr}(H_0|\vec{\theta})}
        &= \frac{\displaystyle\prod_{v=1}^{n}\displaystyle\sum_{i \in S, j = S_j, i \ne j}p_{ij}\times\mathrm{Pr_{SCM}}(GT_{LCA} \ne S_j)}
                {\displaystyle\prod_{v=1}^{n}p_{jj}\times\mathrm{Pr_{SCM}}(GT_{LCA} = S_j)} \\
\end{align}
$$

To provide an intution of this approch, consider the following toy example based on empirical observations. In this example, I've expanded Bayes theorem for each component hypothesis. In the figure below, colors reflect each hypothesis

![Toy example](figs/toy_example.jpg)

While the diagram above represents a case for only two descendent lineages, *equation 1* is compatible with any number of descendent lineages and mutational patterns across these lineages; however, the complexity of system increases dramatically. 

Importantly, this Bayesian framework is compatible with sharing evidence across independently evolving sites. As convergent mutations are not expected to be correlated between any set of lineages, repeated observations favor single origin + technical uncertainty ($H_0$) over repeated convergence ($H_1$). Thus, we can visually represent the effect of accumulated evidence on the posterior ratio as the following (here, colors represent independently evolving somatic variants):

![Aggregating evidence across sites](figs/integrating_across_observations.jpg)

Finally, this schematic can be used to expand *equation 1* to incorporate evidence across independently evolving SNVs.

$$
\begin{align}
\tag{2}
\frac{\mathrm{Pr}(H_1|\vec{\theta})}
     {\mathrm{Pr}(H_0|\vec{\theta})}
= \prod_{m \in \mathrm{SNV_{\mathit{H_1}}}}
    \left(
        \frac{
            \prod\limits_{v=1}^{n}
                \sum\limits_{\substack{i \in S \\ i \neq S_j}}
                    p_{ij} \cdot \mathrm{Pr_{SCM}}(GT_{LCA} \ne S_j)
        }{
            \prod\limits_{v=1}^{n}
                p_{jj} \cdot \mathrm{Pr_{SCM}}(GT_{LCA} = S_j)
        }
    \right)
\end{align}
$$

# 3. Simulation

To generate some expectations of how this method will behave, below are some simulated results across a distribution of parameter values. Among these, we will test the effects of the following:
1. $GT_{LCA}$ **prior probabilities**
2. $v_{LCA} / \sum v_{clade}$, the ratio of branch lengths between LCA and descendents
3. **Observation count** (i.e., number of violations)
4. **Mutation class** (e.g., C>T vs. A>G)

# 3.1 Data generation

In [ ]:
# Load libraries
suppressPackageStartupMessages(
    source("../dev/bin/SCM_functions.R")
)

# Read substitution model
